# Laboratorium 2: Sieci splotowe w klasyfikacji obrazów

Na zbiorze CIFAR-10 porównacie perceptron wielowarstwowy z siecią splotową, policzycie splot ręcznie w numpy i sprawdzicie, że daje to samo co warstwa `Conv2D`. Potem wytrenujecie większą sieć, zobaczycie przeuczenie i porównacie cztery warianty regularyzacji (augmentacja, dropout, L2). Na koniec podejrzycie filtry i mapy cech. Zadania rozszerzone dokładają mały ResNet z blokami rezydualnymi i `GlobalAveragePooling2D`.

Środowisko: Google Colab z GPU (Środowisko wykonawcze, Zmień typ środowiska wykonawczego, T4 GPU). Keras 3 na backendzie TensorFlow, bez dodatkowych instalacji. Na CPU treningi z zadań 4 i 5 trwają bardzo długo.

## Konfiguracja

Wszystkie rozmiary zbiorów i liczby epok są w jednej komórce poniżej.

In [ ]:
# Konfiguracja przebiegu (wartości dla Colaba z GPU T4).
# Jeśli treningi trwają za długo, zmniejszcie N_TRAIN (np. do 20_000) i liczbę epok,
# a we wnioskach zapiszcie, co zmieniliście.
N_TRAIN = 45_000     # obrazy treningowe (maksymalnie 50_000 - N_VAL)
N_VAL = 5_000        # obrazy walidacyjne (ostatnie N_VAL z części treningowej CIFAR-10)
BATCH = 128
EPOKI_MLP = 15
EPOKI_CNN = 15
EPOKI_DUZA = 40      # zadania 4 i 5
EPOKI_RESNET = 30    # zadania rozszerzone

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers
from scipy.signal import convolve2d

RANDOM_STATE = 42
keras.utils.set_random_seed(RANDOM_STATE)
T_START = time.perf_counter()
pd.set_option("display.width", 140)

tf.get_logger().setLevel("ERROR")
print("Keras", keras.__version__, "| backend", keras.backend.backend(), "| TensorFlow", tf.__version__)
gpu = tf.config.list_physical_devices("GPU")
print("GPU:", gpu if gpu else "brak, obliczenia na CPU")

In [ ]:
# Wczytanie CIFAR-10 (przy pierwszym uruchomieniu pobiera około 170 MB)
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train_full = y_train_full.ravel()
y_test = y_test.ravel()
KLASY = ["samolot", "samochód", "ptak", "kot", "jeleń", "pies", "żaba", "koń", "statek", "ciężarówka"]
print("trening:", x_train_full.shape, x_train_full.dtype, "| test:", x_test.shape)
print("zakres pikseli:", x_train_full.min(), "do", x_train_full.max())

In [ ]:
# Gotowe funkcje pomocnicze do rysowania i podsumowań

def pokaz_siatke(obrazy, etykiety, n_kol=10, tytul=None):
    """Siatka obrazów z nazwami klas nad każdym obrazem."""
    n_wier = int(np.ceil(len(obrazy) / n_kol))
    fig, osie = plt.subplots(n_wier, n_kol, figsize=(1.3 * n_kol, 1.5 * n_wier), squeeze=False)
    for ax in osie.ravel():
        ax.axis("off")
    for ax, obr, et in zip(osie.ravel(), obrazy, etykiety):
        ax.imshow(obr)
        ax.set_title(KLASY[int(et)], fontsize=8)
    if tytul:
        fig.suptitle(tytul)
    plt.tight_layout()
    plt.show()


def rysuj_krzywe(historie, metryki=("loss", "accuracy"), tytul=None):
    """historie: słownik nazwa -> History z model.fit. Linia przerywana: trening, ciągła: walidacja."""
    fig, osie = plt.subplots(1, len(metryki), figsize=(6 * len(metryki), 4), squeeze=False)
    for ax, m in zip(osie.ravel(), metryki):
        for i, (nazwa, h) in enumerate(historie.items()):
            h = h.history if hasattr(h, "history") else h
            epoki = np.arange(1, len(h[m]) + 1)
            ax.plot(epoki, h[m], "--", color=f"C{i}", alpha=0.6, label=f"{nazwa}: trening")
            ax.plot(epoki, h["val_" + m], "-", color=f"C{i}", label=f"{nazwa}: walidacja")
        ax.set_xlabel("epoka")
        ax.set_ylabel(m)
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    if tytul:
        fig.suptitle(tytul)
    plt.tight_layout()
    plt.show()


def podsumuj(nazwa, model, historia, czas_s):
    """Jeden wiersz tabeli wyników na podstawie historii treningu."""
    h = historia.history
    return {"model": nazwa, "parametry": model.count_params(), "epoki": len(h["loss"]),
            "train_acc (ost.)": round(h["accuracy"][-1], 4),
            "val_acc (ost.)": round(h["val_accuracy"][-1], 4),
            "val_acc (maks.)": round(max(h["val_accuracy"]), 4),
            "epoka min val_loss": int(np.argmin(h["val_loss"]) + 1),
            "min val_loss": round(min(h["val_loss"]), 4),
            "s/epokę": round(czas_s / len(h["loss"]), 1)}

## Zadanie 1: Dane i baseline MLP

1. Pokażcie siatkę przykładów, po dwa z każdej klasy, z nazwami klas.
2. Znormalizujcie piksele do zakresu [0, 1]. Wydzielcie walidację: ostatnie `N_VAL` obrazów z części treningowej. Zbiór treningowy to pierwsze `N_TRAIN` obrazów. Test odkładacie do zadania 5.
3. Zbudujcie MLP: `Flatten`, `Dense(256, relu)`, `Dense(128, relu)`, `Dense(10, softmax)`. Napiszcie funkcje `kompiluj` (Adam 1e-3, `sparse_categorical_crossentropy`, accuracy) i `trenuj` (fit z walidacją, zwraca historię i czas). Wytrenujcie MLP przez `EPOKI_MLP` epok, wypiszcie liczbę parametrów i dokładność walidacyjną.

Pytania: Skąd bierze się większość parametrów MLP? Jaki wynik dałoby losowe zgadywanie? Co MLP „wie” o tym, że sąsiednie piksele są ze sobą powiązane? Dlaczego walidację wydzielamy z części treningowej, a nie bierzemy zbioru testowego?

In [ ]:
# TODO: wybierzcie po 2 przykłady z każdej klasy (np. np.where(y_train_full == k)[0][:2])
# i pokażcie je funkcją pokaz_siatke(obrazy, etykiety, n_kol=10, tytul=...).

In [ ]:
# TODO: znormalizujcie obrazy do [0, 1] (float32) i wydzielcie zbiory:
#   walidacja: ostatnie N_VAL obrazów z x_train_full, trening: pierwsze N_TRAIN obrazów,
#   test: x_test po normalizacji (użyjecie go tylko raz, w zadaniu 5).
# Wypiszcie kształty zbiorów i liczność klas (np.bincount).
x_tr, y_tr, x_val, y_val, x_te = None, None, None, None, None

In [ ]:
# TODO: zbuduj_mlp(): Input(32, 32, 3), Flatten, Dense(256, relu), Dense(128, relu), Dense(10, softmax).
# TODO: kompiluj(model, lr=1e-3): Adam, loss "sparse_categorical_crossentropy", metryka "accuracy".
# TODO: trenuj(model, epoki) -> (historia, czas_s): model.fit na (x_tr, y_tr),
#       validation_data=(x_val, y_val), batch_size=BATCH, verbose=2; czas zmierzcie time.perf_counter().
# Wytrenujcie MLP (keras.utils.set_random_seed(RANDOM_STATE) przed budową modelu), wypiszcie summary()
# i dodajcie wiersz podsumuj("MLP", model_mlp, hist_mlp, czas_mlp) do listy wyniki.
model_mlp, hist_mlp = None, None
wyniki = []

In [ ]:
# TODO: narysujcie krzywe uczenia MLP: rysuj_krzywe({"MLP": hist_mlp}, tytul=...)

Odpowiedzi:

1. 
2. 
3. 
4. 

## Zadanie 2: Splot ręcznie

Gotowa komórka wybiera jeden obraz (pierwszą ciężarówkę w zbiorze treningowym), zamienia go na skalę szarości i definiuje trzy filtry 3x3: dwa filtry Sobela i rozmycie.

1. Napiszcie `splot2d(obraz, filtr)`: bez dopełniania (`valid`), krok 1, dwie pętle po pozycjach wyniku, w każdej suma iloczynów okna obrazu i filtra.
2. Zastosujcie trzy filtry i pokażcie wyniki funkcją `pokaz_obrazy`.
3. Utwórzcie `layers.Conv2D(3, 3, padding="valid", use_bias=False)`, zbudujcie ją dla wejścia `(1, 32, 32, 1)` i ustawcie `set_weights` tak, by jej trzy filtry były waszymi filtrami. Porównajcie wyniki z numpy (maksymalna różnica bezwzględna).
4. Porównajcie wasz wynik z matematycznym splotem `scipy.signal.convolve2d(obraz, filtr, mode="valid")`, raz z oryginalnym filtrem, raz z filtrem obróconym o 180 stopni (`filtr[::-1, ::-1]`).

Pytania: Gdzie na obrazie filtr Sobela Gy daje duże wartości dodatnie i ujemne, a gdzie Gx? Dlaczego wynik ma rozmiar 30x30? Czym różni się korelacja wzajemna od splotu i dlaczego w sieciach nie ma to praktycznego znaczenia? Dla którego z trzech filtrów oba działania dają to samo?

In [ ]:
idx_obraz = int(np.where(y_train_full == 9)[0][0])          # pierwsza ciężarówka
obraz_rgb = x_train_full[idx_obraz].astype("float32") / 255.0
obraz = obraz_rgb @ np.array([0.299, 0.587, 0.114], dtype="float32")   # skala szarości 32x32

FILTRY = {
    "Sobel Gy (krawędzie poziome)": np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype="float32"),
    "Sobel Gx (krawędzie pionowe)": np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype="float32"),
    "rozmycie 3x3": np.full((3, 3), 1 / 9, dtype="float32"),
}


def pokaz_obrazy(obrazy, tytuly, cmap="gray"):
    """Obrazy jednokanałowe obok siebie, każdy z własną skalą kolorów."""
    fig, osie = plt.subplots(1, len(obrazy), figsize=(3.3 * len(obrazy), 3.3), squeeze=False)
    for ax, o, t in zip(osie.ravel(), obrazy, tytuly):
        im = ax.imshow(o, cmap=cmap)
        ax.set_title(t, fontsize=9)
        ax.axis("off")
        fig.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    plt.show()


print("obraz:", obraz.shape, "| klasa:", KLASY[y_train_full[idx_obraz]])

In [ ]:
# TODO: splot2d(obraz, filtr): wynik ma rozmiar (H - kh + 1, W - kw + 1);
#       wynik[i, j] = suma(obraz[i:i+kh, j:j+kw] * filtr).
# Zastosujcie każdy filtr z FILTRY (słownik nazwa -> wynik, np. wyniki_np) i pokażcie:
# pokaz_obrazy([obraz] + list(wyniki_np.values()), ["oryginał"] + list(wyniki_np))
wyniki_np = None

In [ ]:
# TODO: conv = layers.Conv2D(3, 3, padding="valid", use_bias=False); conv.build((1, 32, 32, 1)).
# Sprawdźcie kształt conv.get_weights()[0] i ułóżcie filtry w tablicę o tym kształcie
# (np.stack(..., axis=-1) i dodatkowa oś na kanał wejściowy). conv.set_weights([jadro]).
# Wynik: keras.ops.convert_to_numpy(conv(obraz[None, :, :, None]))[0]. Porównajcie z numpy.

In [ ]:
# TODO: dla każdego filtra porównajcie (np.allclose, atol=1e-5) splot2d(obraz, f) i splot2d(obraz, f[::-1, ::-1])
# z convolve2d(obraz, f, mode="valid").

Odpowiedzi:

1. 
2. 
3. 
4. 

## Zadanie 3: Pierwsza sieć splotowa

1. Zbudujcie CNN: `Conv2D(32, 3, relu)`, `MaxPooling2D(2)`, `Conv2D(64, 3, relu)`, `MaxPooling2D(2)`, `Conv2D(128, 3, relu)`, `MaxPooling2D(2)`, `Flatten`, `Dense(64, relu)`, `Dense(10, softmax)`. Bez dopełniania (`padding="valid"`, domyślne). Nazwijcie warstwy splotowe `conv1`, `conv2`, `conv3`.
2. Policzcie ręcznie liczbę parametrów `conv1` i `conv2` i porównajcie z `count_params()`.
3. Przewidźcie na kartce rozmiary map cech po każdej warstwie, potem zbudujcie tabelę z `layer.output.shape` i liczbą parametrów każdej warstwy.
4. Wytrenujcie sieć przez `EPOKI_CNN` epok i porównajcie z MLP w tabeli `wyniki` (dokładność, liczba parametrów, czas epoki).

Pytania: Od czego zależy liczba parametrów warstwy `Conv2D`, a od czego nie zależy? Dlaczego CNN ma kilka razy mniej parametrów niż MLP, a działa lepiej? W której warstwie CNN jest najwięcej parametrów? Co dzieje się z rozmiarem przestrzennym i liczbą kanałów w kolejnych warstwach i dlaczego po drugim poolingu jest 6, a nie 6,5?

In [ ]:
# TODO: zbuduj_pierwsza_cnn() według instrukcji (warstwy nazwane conv1, pool1, ..., wyjscie).
# Policzcie ręcznie parametry conv1 i conv2: (k * k * kanały_wejścia + 1) * liczba_filtrów
# i porównajcie z model_cnn.get_layer("conv1").count_params().
# Tabela: dla każdej warstwy (model_cnn.layers[1:]) nazwa, tuple(l.output.shape[1:]), l.count_params().
model_cnn = None

In [ ]:
# TODO: wytrenujcie model_cnn przez EPOKI_CNN epok, dopiszcie wiersz do wyniki,
# narysujcie krzywe MLP i CNN na jednym wykresie i pokażcie pd.DataFrame(wyniki).
hist_cnn = None

Odpowiedzi:

1. 
2. 
3. 
4. 

## Zadanie 4: Przeuczenie

1. Napiszcie `zbuduj_cnn(augmentacja=None, dropout=0.0, l2=0.0, nazwa="cnn")`. Architektura bazowa: trzy etapy, każdy z dwiema warstwami `Conv2D(f, 3, padding="same", relu)` i `MaxPooling2D(2)`, dla f = 32, 64, 128 (nazwy `conv1a`, `conv1b`, ..., `conv3b`), potem `Flatten`, `Dense(256, relu)`, `Dense(10, softmax)`. Parametry `augmentacja`, `dropout` i `l2` przydadzą się w zadaniu 5: na razie funkcja ma działać dla wartości domyślnych, czyli bez regularyzacji.
2. Wytrenujcie ten model przez `EPOKI_DUZA` epok i narysujcie krzywe uczenia.
3. Wskażcie epokę, w której `val_loss` osiąga minimum, i porównajcie dokładność treningową i walidacyjną w tej epoce i na końcu treningu.

Pytania: W której epoce zaczyna się przeuczenie i po czym to widać na obu wykresach? Dlaczego `val_loss` rośnie wyraźnie, a `val_accuracy` spada powoli albo stoi? Jak na moment przeuczenia wpływa liczba obrazów treningowych?

In [ ]:
# TODO: zbuduj_cnn(augmentacja=None, dropout=0.0, l2=0.0, nazwa="cnn") według instrukcji.
# Wskazówki: reg = keras.regularizers.L2(l2) if l2 > 0 else None (przekażcie jako kernel_regularizer);
# x = augmentacja(wejscie) if augmentacja is not None else wejscie;
# Dropout po każdym poolingu i przed ostatnią warstwą, tylko gdy dropout > 0.
# Wytrenujcie model bez regularyzacji przez EPOKI_DUZA epok (ziarno przed budową modelu).
model_duza, hist_duza, czas_duza = None, None, None

In [ ]:
# TODO: krzywe uczenia (rysuj_krzywe), epoka minimum val_loss (np.argmin), train/val accuracy i loss
# w tej epoce i w ostatniej.

Odpowiedzi:

1. 
2. 
3. 

## Zadanie 5: Regularyzacja

1. Napiszcie `zbuduj_augmentacje()`, która zwraca `keras.Sequential` z warstwami `RandomFlip("horizontal")`, `RandomTranslation(0.1, 0.1)` i `RandomRotation(0.05)`. Pokażcie osiem losowych wersji jednego obrazu treningowego (wywołanie z `training=True`).
2. Porównajcie cztery warianty modelu z zadania 4: A bez regularyzacji (to model z zadania 4, nie trenujcie go ponownie), B z augmentacją, C z dropoutem 0,25 i L2 1e-4, D z augmentacją, dropoutem i L2. Warianty B do D trenujcie przez `EPOKI_DUZA` epok, każdy z tym samym ziarnem.
3. Zbudujcie tabelę wyników walidacyjnych (funkcja `podsumuj` plus kolumna z różnicą dokładności treningowej i walidacyjnej) i wykres krzywych `val_loss` oraz `val_accuracy` czterech wariantów.
4. Wybierzcie najlepszy wariant według dokładności walidacyjnej po ostatniej epoce i dopiero dla niego policzcie raz dokładność na zbiorze testowym.

Pytania: Który wariant wygrał i o ile? Jak regularyzacja zmieniła różnicę między dokładnością treningową a walidacyjną? Dlaczego w wariantach z augmentacją lub dropoutem dokładność treningowa bywa niższa od walidacyjnej? Czy krzywe wariantów z regularyzacją wypłaszczyły się, czy dłuższy trening mógłby jeszcze pomóc? Dlaczego test liczymy tylko raz?

In [ ]:
# TODO: zbuduj_augmentacje() -> keras.Sequential([RandomFlip("horizontal"), RandomTranslation(0.1, 0.1),
#       RandomRotation(0.05)], name="augmentacja").
# Pokażcie 8 wersji obrazu x_tr[:1]: aug(x_tr[:1], training=True), keras.ops.convert_to_numpy,
# np.concatenate, np.clip(..., 0, 1), pokaz_siatke(..., [y_tr[0]] * 8, n_kol=8).

In [ ]:
# TODO: słowniki historie_reg i modele_reg z wariantem A (model i historia z zadania 4),
# lista wiersze_reg z podsumuj(...). Dla wariantów B, C, D: keras.utils.set_random_seed(RANDOM_STATE),
# nowa augmentacja (zbuduj_augmentacje()) dla każdego modelu, który jej używa,
# kompiluj(zbuduj_cnn(...)), trenuj(model, EPOKI_DUZA), zapis do słowników i listy.
historie_reg, modele_reg, wiersze_reg = {}, {}, []

In [ ]:
# TODO: tabela_reg = pd.DataFrame(wiersze_reg) z dodatkową kolumną różnicy train_acc - val_acc (ost.).
# Wykres: dwa panele (val_loss i val_accuracy w funkcji epoki), po jednej linii na wariant.
tabela_reg = None

In [ ]:
# TODO: najlepszy = nazwa wariantu z największą val_acc (ost.); dopiero teraz, raz:
# modele_reg[najlepszy].evaluate(x_te, y_test, verbose=0).
najlepszy = None

Odpowiedzi:

1. 
2. 
3. 
4. 
5. 

## Zadanie 6: Co widzi sieć

1. Pokażcie 32 filtry warstwy `conv1a` najlepszego modelu z zadania 5 (funkcja `pokaz_filtry`, wagi z `get_weights()[0]`).
2. Zbudujcie model pomocniczy `keras.Model(model.inputs, [model.get_layer(n).output for n in nazwy])` dla warstw `conv1a`, `conv2b` i `conv3b` i pokażcie mapy cech (funkcja `pokaz_mapy`) dla jednego obrazu walidacyjnego.
3. Policzcie, ile kanałów każdej z tych warstw nie aktywuje się wcale (same zera po ReLU) na 500 obrazach walidacyjnych.

Pytania: Jakie wzorce widać w filtrach pierwszej warstwy (kolor, kierunek krawędzi)? Czym różnią się mapy cech na kolejnych warstwach (rozdzielczość, czytelność dla człowieka)? Co oznaczają kanały, które nigdy się nie aktywują?

In [ ]:
def pokaz_filtry(jadro, n_kol=8, tytul=None):
    """jadro: wagi Conv2D o kształcie (k, k, 3, liczba_filtrów). Każdy filtr skalowany osobno do [0, 1]."""
    n = jadro.shape[-1]
    fig, osie = plt.subplots(int(np.ceil(n / n_kol)), n_kol, figsize=(1.2 * n_kol, 1.2 * np.ceil(n / n_kol)),
                             squeeze=False)
    for i, ax in enumerate(osie.ravel()):
        ax.axis("off")
        if i < n:
            f = jadro[..., i]
            ax.imshow((f - f.min()) / (f.max() - f.min() + 1e-8), interpolation="nearest")
            ax.set_title(str(i), fontsize=7)
    if tytul:
        fig.suptitle(tytul)
    plt.tight_layout()
    plt.show()


def pokaz_mapy(mapa, tytul, n=16, n_kol=8):
    """mapa: aktywacje jednego obrazu o kształcie (wysokość, szerokość, kanały); pokazuje n pierwszych kanałów."""
    n = min(n, mapa.shape[-1])
    fig, osie = plt.subplots(int(np.ceil(n / n_kol)), n_kol, figsize=(1.3 * n_kol, 1.4 * np.ceil(n / n_kol)),
                             squeeze=False)
    for i, ax in enumerate(osie.ravel()):
        ax.axis("off")
        if i < n:
            ax.imshow(mapa[..., i], cmap="viridis")
            ax.set_title(str(i), fontsize=7)
    fig.suptitle(tytul)
    plt.tight_layout()
    plt.show()

In [ ]:
# TODO: model_vis = modele_reg[najlepszy]; wagi conv1a (get_weights()[0]); pokaz_filtry(...).
model_vis = None

In [ ]:
# TODO: ekstraktor = keras.Model(model_vis.inputs, [model_vis.get_layer(n).output for n in nazwy])
# dla nazwy = ["conv1a", "conv2b", "conv3b"]; wybierzcie obraz walidacyjny (np. pierwszy samochód),
# pokażcie go, wypiszcie predykcję modelu i mapy cech: pokaz_mapy(m[0], tytul) dla każdej warstwy.

In [ ]:
# TODO: aktywacje ekstraktora dla x_val[:500]; dla każdej warstwy policzcie kanały,
# których maksimum po wszystkich obrazach i pozycjach wynosi 0.

Odpowiedzi:

1. 
2. 
3. 

# Zadania rozszerzone

## Zadanie 7: Mały ResNet

1. Napiszcie `blok_rezydualny(x, filtry, krok=1, nazwa)`: `Conv2D(3x3, strides=krok, use_bias=False)`, `BatchNormalization`, ReLU, `Conv2D(3x3)`, `BatchNormalization`, dodanie skrótu warstwą `Add` i ReLU. Gdy zmienia się liczba kanałów albo krok jest większy od 1, skrót przechodzi przez `Conv2D(1x1, strides=krok)` z `BatchNormalization`.
2. Napiszcie `zbuduj_resnet(glowa="gap", augmentacja=None)`: wejście (opcjonalnie augmentacja), `Conv2D(32, 3)` z BN i ReLU, trzy etapy po jednym bloku (32 kanały, potem 64 i 128 z krokiem 2), na końcu `GlobalAveragePooling2D` (`glowa="gap"`) albo `Flatten` (`glowa="flatten"`) i `Dense(10, softmax)`.
3. Porównajcie liczbę parametrów obu głów i całych modeli.
4. Wytrenujcie ResNet z GAP i z tą samą augmentacją co w zadaniu 5 przez `EPOKI_RESNET` epok. Porównajcie z najlepszą CNN z zadania 5: parametry, czas epoki, dokładność walidacyjna (także dokładność CNN po tej samej liczbie epok). Na koniec policzcie raz dokładność testową ResNetu.

Pytania: Ile parametrów oszczędza GAP i co traci w zamian? Po co skrót 1x1 przy zmianie liczby kanałów? Jak ResNet wypada przy tej samej liczbie epok co najlepsza CNN i czym może się to tłumaczyć? Jaką rolę grają `BatchNormalization` i połączenia rezydualne w głębszych sieciach?

In [ ]:
# TODO: blok_rezydualny(x, filtry, krok=1, nazwa="blok") i zbuduj_resnet(glowa="gap", augmentacja=None)
# według instrukcji. Porównajcie parametry głowy (warstwa "wyjscie") i całego modelu dla glowa="gap"
# i glowa="flatten" (modele bez treningu) oraz dla CNN z zadania 4 (Flatten + Dense 256).

In [ ]:
# TODO: ziarno, model_resnet = kompiluj(zbuduj_resnet("gap", augmentacja=zbuduj_augmentacje())),
# trening przez EPOKI_RESNET epok.
model_resnet, hist_resnet = None, None

In [ ]:
# TODO: tabela porównawcza najlepszej CNN z zadania 5 i ResNetu (podsumuj plus val_acc CNN po EPOKI_RESNET
# epokach z historie_reg[najlepszy]), krzywe uczenia obu modeli, na końcu raz evaluate ResNetu na x_te.

Odpowiedzi:

1. 
2. 
3. 
4. 

In [ ]:
print(f"Czas wykonania notebooka (od komórki importów): {(time.perf_counter() - T_START) / 60:.1f} min")